## Design Stateful Agentic Loops - Bounded loop with a run record

Make Larkspur's claims agent durable, a loop with a lifecycle and budgets, on Agents v2, that
plans, critiques, forks and resumes, with checkpoints in Azure Cosmos DB.

1. Foundry Agent Service 
2. Agentic Loops
3. Agents v2 
4. Reflection and planning
5. Cosmos DB checkpoints 
6. Fork and resume

Vantor Manufacturing reports a fire at its Hamburg plant. Assessing the claim takes days, across
sessions, and today the claims agent forgets everything between Friday afternoon and Monday morning.
Six chapters make it durable.

        - The production agentic loop
        - Agents v2 and the Responses API
        - Reflection and planning
        - Session state and context
        - Fork and resume
        - Migrate v1 to v2

In [4]:
# libraries
import sys
import time
from pathlib import Path

from common.clients import CLAIM, MODEL, cosmos_db, openai
from common.claims import get_fnol

In [6]:
# sys.path.append(str(Path(__file__).resolve().parents[1]))

ROOT = Path.cwd()

sys.path.append(str(ROOT))


runs = cosmos_db.get_container_client('runs')
BUDGET = {'seconds': 600, 'input_tokens': 100_000}
FNOL = get_fnol(CLAIM)
started = time.monotonic()

In [7]:
response = openai.responses.create(
    model=MODEL, background=True, max_output_tokens=4_000,
    input=f"You are Larkspur Mutual's claims assessor. Outline the assessment steps for this loss, "
    f"and list the documents you would need.\n\n{FNOL}"
)

run = {"id": response.id, "sessionId": CLAIM, "type": "run", "transactions": [response.status]}
print("started", response.id, response.status)

NotFoundError: Error code: 404 - The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.

In [ ]:
while response.status in ("queued", "in_progress"):
    if time.monotonic() - started > BUDGET["seconds"]:
        response = openai.responses.cancel(response.id)
        break
    time.sleep(2)
    response = openai.responses.retrieve(response.id)
    if response.status != run["transitions"][-1]:
        run["transitions"].append(response.status)

In [ ]:
run["status"] = response.status
run["seconds"] = round(time.monotonic() - started, 1)
if response.status == "completed":
    run["input_tokens"] = response.usage.input_tokens
    run["output_tokens"] = response.usage.output_tokens
    run["within_budget"] = run["input_tokens"] <= BUDGET["input_tokens"]
elif response.status == "incomplete":
    run["reason"] = response.incomplete_details.reason      # raise max_output_tokens or split the step
elif response.status == "failed":
    run["reason"] = response.error.code                     # rate_limit_exceeded: back off; invalid_prompt: fix it
runs.upsert_item(run)

In [ ]:
print(" -> ".join(run["transitions"]), run.get("reason", ""))
if response.status == "completed":
    print(response.output_text[:800])